# CarRacing: PPO + IMPALA-style residual CNN

This notebook is self-contained: upload this `.ipynb` to Colab; no repository clone or `src/` imports are needed. It trains from scratch. **The journal is not modified.**

We implement the environment wrappers, image encoder, evaluation and checkpoint workflow here. **Stable-Baselines3 (SB3) implements PPO**, including the rollout buffer, GAE, clipped loss, minibatches and optimizer. This is a usable baseline while we study those pieces; it does not fill the empty handwritten PPO modules in the repository.

The residual CNN is a comparison to run after the basic CNN. Neither is a proven winner on this competition. We keep PPO and evaluation the same so the image encoder is the main intended difference.

Run top to bottom. First run the short pipeline check, inspect its output and video, then explicitly enable long training. A short run is not expected to produce a competent driver.

This version matches the basic-CNN **gSDE recovery** settings: `use_sde=True`, noise resampling every four decisions, `log_std_init=-2`, and `ent_coef=0`. The encoder is the intended configuration difference. Use a fresh IMPALA run; a basic-CNN checkpoint cannot initialize this encoder. Equal transition budgets and measured wall-clock times are both relevant. This is not the full IMPALA algorithm and is not a proven repair.


## 1. Open in Colab and choose a GPU

1. Open https://colab.research.google.com/ and choose **File → Upload notebook**. Select this file.
2. Choose **Runtime → Change runtime type → Hardware accelerator → GPU**. Select **T4** if available, or another available NVIDIA GPU. Leave runtime language as Python 3. Do not choose TPU for this notebook.
3. Click Save, then Connect. Changing the runtime later can erase the current session memory, so choose the GPU first.
4. Execute cells using the triangle button or Shift+Enter. Read the output before advancing.
5. Start with the basic CNN notebook. Run the IMPALA notebook separately afterward; do not train both simultaneously on one runtime. Each uses its own output folder.

GPU type, availability and session duration are not guaranteed. Physics still runs on CPU, so a GPU does not accelerate every part. [Colab FAQ](https://research.google.com/colaboratory/faq.html).


## 2. Install the dependencies

`%pip` installs into this notebook's Python environment. The first line supplies SWIG for Box2D builds. We preserve a compatible preinstalled CUDA-enabled PyTorch; there is no `--upgrade` flag. No system-wide CUDA installation is needed.

Run this installation cell once per new runtime. If Colab asks to restart the session, do so, then continue with the imports below. Installation failure should be resolved before starting training.


In [ ]:
%pip install -q "swig==4.*"
%pip install -q "gymnasium[box2d]==1.3.0" "stable-baselines3==2.9.0" "torch>=2.8,<3.0" "numpy>=1.26,<3" "pillow>=10,<13" "matplotlib>=3.8,<4" "tensorboard>=2.16,<3" "imageio[ffmpeg]>=2.34,<3" "tqdm>=4.66,<5" "rich>=13"


In [ ]:
import os
# Headless SDL: Colab has no desktop window. Images/video are displayed inline.
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"

import json
import sys
import time
import importlib.metadata as metadata
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import imageio.v2 as imageio
import gymnasium as gym
import torch
from torch import nn
import stable_baselines3 as sb3
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.vec_env import DummyVecEnv, VecTransposeImage, VecFrameStack
from IPython.display import Video, display

# Avoid CPU thread oversubscription for these fairly small networks.
torch.set_num_threads(2)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Python:", sys.version.split()[0])
print("Gymnasium:", gym.__version__, "SB3:", sb3.__version__, "Torch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU is fine for setup checks. Select a GPU runtime before long training.")
assert gym.__version__ == "1.3.0", "Use the pinned Gymnasium version for these checks"


## 3. Mount Drive and choose an experiment folder

The notebook file and the training model are different things. Saving the notebook does not save its RAM or model weights. `/content` is temporary; Google Drive keeps checkpoints after a disconnect.

Use `USE_DRIVE=True` in Colab and approve your own Drive mount. Local testing can use `False`. `EXPERIMENT_NAME` identifies one run. Keep its name to resume; choose a new name for a new training seed or changed settings. The CNN and IMPALA notebooks have different architecture folders automatically.

The new default folder is `impala_cnn/gsde_comparison_01`. Keep `basic_cnn/gsde_recovery_01` intact. Older IMPALA `run_01` checkpoints used different exploration; do not resume them under this configuration.


In [ ]:
USE_DRIVE = True
ARCHITECTURE = "impala_cnn"
EXPERIMENT_NAME = "gsde_comparison_01"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/max-verstappen")
else:
    OUTPUT_ROOT = Path.cwd() / "racing_outputs"
RUN_DIR = OUTPUT_ROOT / ARCHITECTURE / EXPERIMENT_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = Path("/content/racing_logs") / ARCHITECTURE / EXPERIMENT_NAME if USE_DRIVE else RUN_DIR / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
print("Persistent models/results:", RUN_DIR)
print("TensorBoard logs for this session:", LOG_DIR)


## 4. Fix the track groups and experiment settings

**One shared policy learns all training tracks.** At every episode reset, choose one of 100 training seeds uniformly, with replacement. We do not finish learning one track before moving to the next. The seed numbers are arbitrary identifiers, not difficulty scores. One hundred is a starting experiment, not proof of sufficient diversity.

Training seeds are `1000–1099`; validation is `10000–10019`; final test is `20000–20019`; the public competition seeds are `202`, `42`, `203`. We exclude all evaluation groups from training. The fixed final test is used only after model selection. The organizer's mystery track remains unknown.

`run_seed` controls initialization and sampling randomness; a track seed generates a road. These are different jobs. This baseline uses a fixed training pool so both CNN experiments can be compared; if training improves but validation does not, broader training diversity is an experiment to try.

PPO settings below are starting values, not tuned winning values:
- `n_envs=2`: two environment instances, batched for the network.
- `n_steps=512`: each instance contributes 512 decisions; a rollout contains 1024 transitions.
- `batch_size=128`: eight minibatches per rollout.
- `n_epochs=5`: five shuffled passes over that fresh rollout.
- `learning_rate=3e-4`: optimizer step-size setting.
- `gamma=0.99`, `gae_lambda=0.95`: discount and GAE weighting per decision transition.
- `clip_range=0.2`: clips the policy probability-ratio objective, not the steering wheel.
- `ent_coef=0`: no additional entropy bonus; exploration remains active. `vf_coef=0.5`: critic-loss weight.
- `use_sde=True`: exploration depends on actor features.
- `sde_sample_freq=4`: resample noise weights every four collection decisions; actions can still change between resamplings.
- `log_std_init=-2`: initialize noise-weight scales near 0.135; effective action spread also depends on features.
- `max_grad_norm=0.5`: gradient clipping; `target_kl=0.03`: approximate-KL early stopping.

Changing action repeat changes the amount of real time represented by a transition and therefore the effective discount horizon. Keep it fixed during the initial encoder comparison.


In [ ]:
TRAINING_SEEDS = tuple(range(1000, 1100))
VALIDATION_SEEDS = tuple(range(10000, 10020))
TEST_SEEDS = tuple(range(20000, 20020))
PUBLIC_SEEDS = (202, 42, 203)
assert not set(TRAINING_SEEDS) & set(VALIDATION_SEEDS + TEST_SEEDS + PUBLIC_SEEDS)

CONFIG = {
    "architecture": ARCHITECTURE,
    "run_seed": 7,
    "training_seeds": list(TRAINING_SEEDS),
    "image_size": 84, "frame_stack": 4, "action_repeat": 2,
    "n_envs": 2, "n_steps": 512, "batch_size": 128, "n_epochs": 5,
    "learning_rate": 3e-4, "gamma": 0.99, "gae_lambda": 0.95,
    "clip_range": 0.2, "ent_coef": 0.0, "vf_coef": 0.5,
    "use_sde": True, "sde_sample_freq": 4, "log_std_init": -2.0,
    "max_grad_norm": 0.5, "target_kl": 0.03,
    "features_dim": 256,
}
assert CONFIG["n_steps"] * CONFIG["n_envs"] % CONFIG["batch_size"] == 0

config_path = RUN_DIR / "config.json"
if config_path.exists():
    previous_config = json.loads(config_path.read_text())
    if previous_config != CONFIG:
        raise ValueError("Settings differ from this folder's run. Choose a NEW EXPERIMENT_NAME.")
config_path.write_text(json.dumps(CONFIG, indent=2))
versions = "\n".join(sorted(f"{dist.metadata['Name']}=={dist.version}" for dist in metadata.distributions())) + "\n"
(RUN_DIR / "requirements_runtime.txt").write_text(versions)
print(json.dumps(CONFIG, indent=2))


## 5. Inspect what the environment really returns

The agent controls `[steering, gas, brake]`. Steering ranges from -1 to 1; gas and brake from 0 to 1. The picture has 96 rows, 96 columns and three color channels. `uint8` stores integer pixel intensities from 0 to 255; it is not a neural-network type.

`reset(seed=...)` starts a track. `step(action)` advances the simulator and returns a new image, reward, `terminated`, `truncated`, and metadata. Termination can mean a finish or a failure. A time limit is truncation. Neither reset nor step trains a model.

This preview uses one training track, not a held-out test track. It closes its environment afterward. Using `render_mode="human"` would try to open a desktop window, which is not the Colab display workflow.


In [ ]:
preview = gym.make("CarRacing-v3", continuous=True, max_episode_steps=3000)
try:
    image, info = preview.reset(seed=TRAINING_SEEDS[0])
    print("Observation:", image.shape, image.dtype)
    print("Action bounds:", preview.action_space.low, preview.action_space.high)
    assert image.shape == (96, 96, 3) and image.dtype == np.uint8
    plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.title("Raw image: the policy's source of information")
    plt.axis("off")
    plt.show()
finally:
    preview.close()


## 6. Build preprocessing and track sampling

The wrapper order is deliberate:

`CarRacing + 3000-step TimeLimit → track sampler → repeat action → grayscale/resize → Monitor → vector batch → transpose → frame stack`.

`TrackSampler` chooses training tracks randomly or evaluation tracks in a fixed order. Seeds are logging metadata, never policy inputs. Our explicit pool controls separation; it is different from passing a track seed to the base environment directly.

`ActionRepeat` sends one decision to at most two simulator steps, sums their rewards and stops immediately at an episode boundary. Its counter gives actual simulator steps for evaluation. The inner time limit still allows only 3000 raw steps.

`GrayResize` converts to grayscale and 84×84 while preserving uint8 and a singleton channel. SB3 later divides by 255, exactly once. No extra normalization belongs in these wrappers.

`VecTransposeImage` puts channels first for PyTorch. `VecFrameStack` builds `(n_envs, 4, 84, 84)`. SB3 zero-pads missing history at reset and clears history between episodes. With repeat 2, consecutive stacked frames are two simulator steps apart.

SB3's vector API differs from Gymnasium: reset returns just observations; step returns `(observations, rewards, dones, infos)` and automatically resets finished episodes. Terminal images remain in `info['terminal_observation']`, which SB3 uses for time-limit bootstrapping.


In [ ]:
class TrackSampler(gym.Wrapper):
    """Choose tracks from one explicit pool; the policy never receives the seed."""
    def __init__(self, env, seeds, random_order=True, sampler_seed=7):
        super().__init__(env)
        self.seeds = tuple(int(seed) for seed in seeds)
        if not self.seeds:
            raise ValueError("The track pool cannot be empty")
        self.random_order = random_order
        self.rng = np.random.default_rng(sampler_seed)
        self.index = 0
        self.track_seed = None

    def reset(self, *, seed=None, options=None):
        # Here seed controls the SAMPLER. The chosen track seed is passed on
        # to CarRacing. SB3 seeds each worker once before its first reset.
        if seed is not None:
            self.rng = np.random.default_rng(seed)
            self.index = 0
        if self.random_order:
            self.track_seed = int(self.rng.choice(self.seeds))
        else:
            self.track_seed = self.seeds[self.index % len(self.seeds)]
            self.index += 1
        observation, info = self.env.reset(seed=self.track_seed, options=options)
        return observation, {**info, "track_seed": self.track_seed}

    def step(self, action):
        observation, reward, terminated, truncated, info = self.env.step(action)
        return observation, reward, terminated, truncated, {
            **info, "track_seed": self.track_seed,
        }


class ActionRepeat(gym.Wrapper):
    """Repeat a decision, sum rewards, and count actual simulator steps."""
    def __init__(self, env, repeat=2):
        super().__init__(env)
        if repeat < 1:
            raise ValueError("repeat must be positive")
        self.repeat = repeat
        self.raw_steps = 0

    def reset(self, **kwargs):
        self.raw_steps = 0
        return self.env.reset(**kwargs)

    def step(self, action):
        total_reward = 0.0
        for _ in range(self.repeat):
            observation, reward, terminated, truncated, info = self.env.step(action)
            self.raw_steps += 1
            total_reward += float(reward)
            # Never continue stepping after either kind of episode ending.
            if terminated or truncated:
                break
        return observation, total_reward, terminated, truncated, {
            **info, "raw_steps": self.raw_steps,
        }


class GrayResize(gym.ObservationWrapper):
    """RGB -> resized grayscale; keep uint8 until SB3 normalizes it once."""
    def __init__(self, env, size=84):
        super().__init__(env)
        self.size = size
        self.observation_space = gym.spaces.Box(
            low=0, high=255, shape=(size, size, 1), dtype=np.uint8,
        )

    def observation(self, observation):
        image = Image.fromarray(observation).convert("L")
        image = image.resize((self.size, self.size), Image.Resampling.BILINEAR)
        return np.asarray(image, dtype=np.uint8)[..., None].copy()


def make_single_env(seeds, random_order, sampler_seed, render_mode=None):
    # TimeLimit is INSIDE action repetition: 3000 means raw simulator steps.
    env = gym.make(
        "CarRacing-v3", continuous=True, max_episode_steps=3000,
        render_mode=render_mode,
    )
    env = TrackSampler(env, seeds, random_order, sampler_seed)
    env = ActionRepeat(env, repeat=CONFIG["action_repeat"])
    env = GrayResize(env, size=CONFIG["image_size"])
    return Monitor(env)


def build_vec_env(seeds, n_envs=1, random_order=True, render_mode=None):
    factories = []
    for rank in range(n_envs):
        # Bind rank now, rather than letting every factory use the last rank.
        def factory(rank=rank):
            return make_single_env(
                seeds, random_order, CONFIG["run_seed"] + rank, render_mode,
            )
        factories.append(factory)
    # DummyVecEnv runs its workers sequentially on the CPU. It batches their
    # observations for the network; it does not make the physics run on a GPU.
    env = DummyVecEnv(factories)
    env = VecTransposeImage(env)  # (84,84,1) -> (1,84,84)
    env = VecFrameStack(env, n_stack=CONFIG["frame_stack"], channels_order="first")
    env.seed(CONFIG["run_seed"])
    return env


In [ ]:
# Inspect the policy boundary before training anything.
probe = build_vec_env(TRAINING_SEEDS, n_envs=2)
try:
    stacked = probe.reset()
    assert stacked.shape == (2, 4, 84, 84)
    assert stacked.dtype == np.uint8
    assert np.all(stacked[:, :3] == 0), "Missing history should be zero at reset"
    actions = np.array([[0.0, 0.2, 0.0]] * 2, dtype=np.float32)
    next_stacked, rewards, dones, infos = probe.step(actions)
    assert all(item["raw_steps"] == 2 for item in infos)
    assert all(item["track_seed"] in TRAINING_SEEDS for item in infos)
    print("Shape/dtype:", stacked.shape, stacked.dtype)
    print("Sampled tracks:", [item["track_seed"] for item in infos])
    print("Reward after one decision:", rewards)
    fig, axes = plt.subplots(1, 4, figsize=(10, 3))
    for i, axis in enumerate(axes):
        axis.imshow(next_stacked[0, i], cmap="gray", vmin=0, vmax=255)
        axis.set_title(f"History slot {i}")
        axis.axis("off")
    plt.show()
finally:
    probe.close()


## 7. Implement the IMPALA-style residual CNN

A convolution slides learned filters across the image. Channels are learned feature maps; they are not necessarily human-readable road detectors. ReLU applies `max(0, x)` so the network can learn nonlinear relationships. Flatten changes the layout, not the learned meaning. The final linear layer produces 256 features for the actor and critic.

There are three stages with 16, 32 and 32 channels. Each stage has a convolution, spatial max-pooling and two residual blocks. At 84×84 input, stage outputs have spatial sizes 42×42, 21×21 and 11×11. Each residual block learns F(x) and adds x: the shortcut provides a direct path for information and gradients. The skip addition requires matching shapes. This encoder has 15 convolutional layers (three stages × five convolutions), six residual blocks and one final linear projection. More layers do not automatically mean a better or faster driver.

The dummy tensor in `__init__` determines the flatten size without hard-coding it. `torch.no_grad()` avoids building a gradient graph for that shape calculation. `forward()` defines the actual image-to-feature computation. SB3's preprocessing handles normalization before calling it.

Only the image encoder is borrowed from IMPALA; the learning algorithm remains PPO. This is not a world model and does not predict future frames. A fair comparison must account for its different runtime as well as its validation scores.


In [ ]:
class ResidualBlock(nn.Module):
    """Learn a change F(x), then return x + F(x). Channels stay the same."""
    def __init__(self, channels):
        super().__init__()
        self.change = nn.Sequential(
            nn.ReLU(), nn.Conv2d(channels, channels, 3, padding=1),
            nn.ReLU(), nn.Conv2d(channels, channels, 3, padding=1),
        )

    def forward(self, x):
        return x + self.change(x)


class ImpalaStage(nn.Module):
    """Change channel count, downsample, then apply two residual blocks."""
    def __init__(self, input_channels, output_channels):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(input_channels, output_channels, 3, padding=1),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            ResidualBlock(output_channels), ResidualBlock(output_channels),
        )

    def forward(self, x):
        return self.layers(x)


class ImpalaCNN(BaseFeaturesExtractor):
    """Small IMPALA-style image encoder, used WITH PPO, not IMPALA learning."""
    def __init__(self, observation_space, features_dim=256):
        super().__init__(observation_space, features_dim)
        channels = observation_space.shape[0]
        self.stages = nn.Sequential(
            ImpalaStage(channels, 16),
            ImpalaStage(16, 32),
            ImpalaStage(32, 32),
            nn.ReLU(), nn.Flatten(),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, *observation_space.shape)
            flattened_size = self.stages(dummy).shape[1]
        self.projection = nn.Sequential(
            nn.Linear(flattened_size, features_dim), nn.ReLU(),
        )

    def forward(self, observations):
        # The image has already been normalized by SB3, exactly once.
        return self.projection(self.stages(observations))


Encoder = ImpalaCNN


In [ ]:
observation_space = gym.spaces.Box(0, 255, shape=(4, 84, 84), dtype=np.uint8)
encoder = Encoder(observation_space, features_dim=CONFIG["features_dim"])
with torch.no_grad():
    features = encoder(torch.zeros(2, 4, 84, 84))
assert features.shape == (2, 256)
print(encoder)
print("Feature shape:", features.shape)
print("Encoder parameters:", sum(p.numel() for p in encoder.parameters()))
del encoder


## 8. Understand PPO's actor, critic and action distribution

The shared CNN supplies 256 features to two separate MLP heads. Each head has a 64-unit Tanh hidden layer. The actor produces three Gaussian means. gSDE learns a matrix of noise scales over the 64 actor features and three controls. It samples an exploration matrix and reuses it for four vector steps, while changes in the state features still change the resulting action noise. This provides more temporal structure than drawing independent action noise at every decision. The effective action standard deviation depends on the current features: `exp(-2)` is a noise-weight scale here, not a fixed standard deviation for each control. The critic produces one value estimate per observation. A stochastic Gaussian may sample outside the legal action range: SB3 stores the original sample/log probability for PPO and clips the action sent to the game. Deterministic prediction uses the mean and the same bounds. We do not add a new tanh action transformation at evaluation.

**What SB3 handles internally:** collect a fresh rollout with the old policy; compute GAE and return targets; normalize advantages; perform shuffled minibatch updates using the clipped policy loss, value loss and entropy term; clip gradients; collect fresh experience. Old values, probabilities and targets are fixed during each update. This notebook does not claim to implement those internals from scratch.

GAE starts from `delta = reward + gamma × next_value − value`. It combines later errors with `gamma × lambda`, but does not propagate advantages across episode boundaries. True termination has zero future value. Time-limit truncation normally bootstraps from the final observation, not the auto-reset image. Ending a rollout midway through a race also needs a next-value estimate.

The actor objective is `mean(min(ratio × advantage, clip(ratio, 1-epsilon, 1+epsilon) × advantage))`, where `ratio = exp(new_log_prob − old_log_prob)`. We minimize its negative. Clipping limits the incentive for certain probability-ratio changes; it is not a guarantee of safe driving or bounded total network change.

With `ent_coef=0`, entropy is still logged but its contribution to the loss is zero. Both actor and critic objectives can update the shared residual CNN.


In [ ]:
# A numerical illustration only: this cell does NOT update the real policy.
old_log_prob = torch.tensor([-1.0, -1.0])
new_log_prob = old_log_prob + torch.log(torch.tensor([1.4, 0.6]))
advantage = torch.tensor([2.0, -2.0])
ratio = (new_log_prob - old_log_prob).exp()
unclipped = ratio * advantage
clipped = ratio.clamp(0.8, 1.2) * advantage
actor_loss = -torch.minimum(unclipped, clipped).mean()
print("Ratios:", ratio.tolist())
print("Unclipped terms:", unclipped.tolist())
print("Clipped terms:", clipped.tolist())
print("Actor loss:", actor_loss.item())
# For both examples the favorable ratio change is already beyond clipping.
# This example illustrates the objective, not a complete optimizer update.


## 9. Create a new model or resume a checkpoint

Leave `RESUME_FROM=""` for the first run. To resume after a disconnected runtime, rerun the preceding setup/definition cells with the same folder/settings, set `RESUME_FROM` to `str(RUN_DIR / "latest.zip")`, then execute this cell. Loading restores the policy and optimizer, not the exact simulator state or uninterrupted random-number sequence.

A checkpoint from the other encoder is not interchangeable. The embedded experiment configuration is checked. If this folder already contains a checkpoint, the cell asks you to resume or choose a new experiment name instead of silently overwriting the run.

The model uses the GPU when available. The original reward is unchanged. The 1024-transition rollout size counts policy decisions across both environments, not episodes and not raw simulator frames.

On the **first IMPALA run**, leave `RESUME_FROM=""`. After a runtime restart, first rerun setup and definitions, then set `RESUME_FROM=str(RUN_DIR / "latest.zip")` before running the loading cell. Existing basic-CNN backups belong to the other architecture and must not be loaded here.


In [ ]:
RESUME_FROM = ""  # Example: str(RUN_DIR / "latest.zip")

# Close a previously constructed training env if this cell is rerun.
if "train_env" in globals():
    train_env.close()
train_env = build_vec_env(TRAINING_SEEDS, n_envs=CONFIG["n_envs"])
if RESUME_FROM:
    model = PPO.load(RESUME_FROM, env=train_env, device=DEVICE)
    if getattr(model, "experiment_config", None) != CONFIG:
        raise ValueError("Checkpoint settings/encoder do not match this experiment")
    print("Resumed at", model.num_timesteps, "policy transitions")
else:
    if (RUN_DIR / "latest.zip").exists():
        raise FileExistsError("A run exists here. Set RESUME_FROM or use a new EXPERIMENT_NAME.")
    model = PPO(
        "CnnPolicy", train_env, device=DEVICE, seed=CONFIG["run_seed"],
        policy_kwargs={
            "features_extractor_class": Encoder,
            "features_extractor_kwargs": {"features_dim": CONFIG["features_dim"]},
            "net_arch": {"pi": [64], "vf": [64]},
            "share_features_extractor": True,
            "normalize_images": True,
            # Match the recovery policy: initialize gSDE noise-weight scales.
            "log_std_init": CONFIG["log_std_init"],
        },
        # These PPO arguments activate gSDE, not just document it in CONFIG.
        use_sde=CONFIG["use_sde"], sde_sample_freq=CONFIG["sde_sample_freq"],
        n_steps=CONFIG["n_steps"], batch_size=CONFIG["batch_size"],
        n_epochs=CONFIG["n_epochs"], learning_rate=CONFIG["learning_rate"],
        gamma=CONFIG["gamma"], gae_lambda=CONFIG["gae_lambda"],
        clip_range=CONFIG["clip_range"], ent_coef=CONFIG["ent_coef"],
        vf_coef=CONFIG["vf_coef"], max_grad_norm=CONFIG["max_grad_norm"],
        target_kl=CONFIG["target_kl"], tensorboard_log=str(LOG_DIR), verbose=1,
    )
    model.experiment_config = CONFIG.copy()
print(model.policy)
print("Total policy parameters:", sum(p.numel() for p in model.policy.parameters()))
# The policy printout can show several aliases for the SAME shared CNN.
assert model.policy.pi_features_extractor is model.policy.vf_features_extractor
print("Actor and critic share one CNN:", True)
with torch.no_grad():
    dummy_obs, _ = model.policy.obs_to_tensor(np.zeros((1, 4, 84, 84), dtype=np.uint8))
    distribution = model.policy.get_distribution(dummy_obs).distribution
    dummy_value = model.policy.predict_values(dummy_obs)
    print("Actor mean/std shapes:", distribution.mean.shape, distribution.stddev.shape)
    print("Critic value shape:", dummy_value.shape)
# These zero-image outputs check interfaces, not driving ability.


## 10. Evaluate actual finishes and save useful checkpoints

`evaluate_tracks` creates separate environments with fixed track seeds. It uses deterministic actions and never calls `learn()`. It reads only completion/timing metadata for scoring; the policy still receives images only.

A real finish is detected through Gymnasium 1.3.0's `info['lap_finished']`, not merely `done=True`. We count the raw steps from our action-repeat wrapper and divide by 50 simulator steps per second. A finish uses its time; a failure/time-out receives 70 seconds. These are local challenge-style measurements, to be checked against the organizer's exact evaluator. No hidden coordinates or track geometry are read.

`RacingCallback` saves `latest.zip` every 25,000 policy transitions and checks five fixed validation tracks every 100,000. `best_validation.zip` minimizes mean penalized time, breaking equal-time ties using higher time-adjusted raw reward (`reward + 0.1 × raw_steps`) and retaining the earlier checkpoint when both are equal. The tie-breaker uses the original reward and retains any off-map penalty; it is a progress proxy, not an exact tile count. This avoids preferring an early crash just because it accrued fewer time penalties. If every checkpoint fails all tracks, they tie at 70 seconds: that means there is still no evidence of successful lap completion, even if the progress proxy improves. The 20-track validation cell later provides a broader audit.

SB3's vector environment auto-resets after a finish. The evaluator uses the ending episode's metadata from `infos`, rather than counting the new episode. Optional video rendering is streamed to disk and does not affect the actor's inputs.


In [ ]:
def evaluate_tracks(policy, seeds, split="validation", video_path=None):
    """Evaluate fixed tracks without updating weights; optionally stream video."""
    pools = {
        "validation": VALIDATION_SEEDS, "test": TEST_SEEDS,
        "public": PUBLIC_SEEDS, "training": TRAINING_SEEDS,
    }
    if split not in pools or not set(seeds).issubset(set(pools[split])):
        raise ValueError("Evaluation seeds must belong to the requested split")
    if video_path is not None and len(seeds) != 1:
        raise ValueError("Record one track at a time")
    rows = []
    for seed in seeds:
        env = build_vec_env(
            (seed,), n_envs=1, random_order=False,
            render_mode="rgb_array" if video_path else None,
        )
        writer = None
        try:
            observation = env.reset()
            total_reward = 0.0
            if video_path:
                # Stream to disk rather than accumulating hundreds of MB in RAM.
                writer = imageio.get_writer(
                    str(video_path), fps=50 / CONFIG["action_repeat"],
                )
            while True:
                action, _ = policy.predict(observation, deterministic=True)
                observation, rewards, dones, infos = env.step(action)
                total_reward += float(rewards[0])
                if writer and not dones[0]:
                    frame = env.envs[0].render()
                    frame = Image.fromarray(frame).resize((480, 320))
                    writer.append_data(np.asarray(frame))
                if dones[0]:
                    info = infos[0]  # Metadata belongs to the episode that ended.
                    # SB3 auto-resets: observation is now the NEW episode's image.
                    # The final image, if needed, is info['terminal_observation'].
                    if not info.get("TimeLimit.truncated", False) and "lap_finished" not in info:
                        raise RuntimeError("This Gymnasium version lacks lap completion metadata")
                    finished = bool(info.get("lap_finished", False))
                    steps = int(info["raw_steps"])
                    lap_seconds = steps / 50.0 if finished else None
                    rows.append({
                        "split": split, "track_seed": int(seed),
                        "finished": finished, "raw_steps": steps,
                        "lap_seconds": lap_seconds,
                        "penalized_seconds": lap_seconds if finished else 70.0,
                        "reward": total_reward,
                    })
                    break
        finally:
            if writer is not None:
                writer.close()
            env.close()
    return rows


def summarize(rows):
    finishes = [row["lap_seconds"] for row in rows if row["finished"]]
    return {
        "tracks": len(rows),
        "completion_rate": sum(row["finished"] for row in rows) / len(rows),
        "mean_penalized_seconds": float(np.mean([row["penalized_seconds"] for row in rows])),
        "mean_finishing_seconds": float(np.mean(finishes)) if finishes else None,
        "mean_reward": float(np.mean([row["reward"] for row in rows])),
    }


def save_checkpoint(policy, name):
    """Write a complete SB3 model, including optimizer state, to Drive/local disk."""
    final_path = RUN_DIR / f"{name}.zip"
    temporary_path = RUN_DIR / f".{name}.tmp.zip"
    policy.save(str(temporary_path))
    os.replace(temporary_path, final_path)
    return final_path


class RacingCallback(BaseCallback):
    """Checkpoint and validate by policy-transition count, not callback calls."""
    def __init__(self, save_every=25000, evaluate_every=100000):
        super().__init__()
        self.save_every = save_every
        self.evaluate_every = evaluate_every
        self.next_save = None
        self.next_evaluation = None
        self.history_path = RUN_DIR / "validation_history.json"
        self.history = json.loads(self.history_path.read_text()) if self.history_path.exists() else []
        # All-DNF checkpoints tie at 70 seconds. For ties, compare original
        # reward with its elapsed-time penalty added back. Raw reward alone
        # could favor crashing early simply because fewer time penalties accrue.
        self.best_key = min(
            (self.selection_key(item["summary"], item["tracks"]) for item in self.history),
            default=(float("inf"), float("inf")),
        )

    @staticmethod
    def selection_key(summary, rows):
        # This proxy retains the environment's off-map penalty. It is NOT an
        # exact tile count and assumes unchanged CarRacing reward (-0.1/raw step).
        progress_proxy = float(np.mean([
            row["reward"] + 0.1 * row["raw_steps"] for row in rows
        ]))
        return (summary["mean_penalized_seconds"], -progress_proxy)

    def _on_training_start(self):
        steps = self.model.num_timesteps
        self.next_save = (steps // self.save_every + 1) * self.save_every
        self.next_evaluation = (steps // self.evaluate_every + 1) * self.evaluate_every

    def _on_step(self):
        # num_timesteps increases by n_envs at each vector-environment step.
        steps = self.model.num_timesteps
        if steps >= self.next_save:
            save_checkpoint(self.model, "latest")
            self.next_save += self.save_every
        if steps >= self.next_evaluation:
            # Five fixed tracks keep frequent evaluation affordable.
            rows = evaluate_tracks(self.model, VALIDATION_SEEDS[:5])
            summary = summarize(rows)
            item = {"timesteps": steps, "summary": summary, "tracks": rows}
            self.history.append(item)
            self.history_path.write_text(json.dumps(self.history, indent=2))
            score = summary["mean_penalized_seconds"]
            self.logger.record("validation/penalized_seconds", score)
            self.logger.record("validation/completion_rate", summary["completion_rate"])
            print(f"\nValidation at {steps:,} transitions: {summary}")
            selection_key = self.selection_key(summary, rows)
            if selection_key < self.best_key:
                self.best_key = selection_key
                save_checkpoint(self.model, "best_validation")
                (RUN_DIR / "best_validation_score.json").write_text(json.dumps(item, indent=2))
            self.next_evaluation += self.evaluate_every
        return True


## 11. Short pipeline check, timing and reload check

This runs **one rollout, about 1024 transitions** with the current settings. It should exercise collection, GAE, gradients, checkpoint writing and reloading. It is intentionally too short to judge driving skill. Learning can be slow or look random here.

On a resumed run, set `RUN_SMOKE=False` if the pipeline already works. This short update is included in the model's training history; the long-run target below uses the cumulative count. The timing includes initialization effects and is only a rough first estimate. A 50,000-transition segment later gives a more representative throughput measurement.

The reload comparison checks that the saved model produces the same deterministic action for the same observation. It does not claim to reproduce an interrupted simulator state.


In [ ]:
RUN_SMOKE = True
if RUN_SMOKE:
    started = time.perf_counter()
    initial_steps = model.num_timesteps
    model.learn(
        total_timesteps=CONFIG["n_steps"] * CONFIG["n_envs"],
        reset_num_timesteps=False, tb_log_name="ppo", progress_bar=False,
    )
    elapsed = time.perf_counter() - started
    added_steps = model.num_timesteps - initial_steps
    print(f"Added {added_steps:,} policy transitions in {elapsed:.1f}s")
    print(f"Rough throughput: {added_steps / elapsed:.1f} transitions/s")
    print(f"At that speed, 1M transitions: {1_000_000 / (added_steps / elapsed) / 3600:.2f} hours, before extra evaluation")
    checkpoint = save_checkpoint(model, "latest")
    check_env = build_vec_env((VALIDATION_SEEDS[0],), random_order=False)
    try:
        check_obs = check_env.reset()
        before, _ = model.predict(check_obs, deterministic=True)
        loaded = PPO.load(str(checkpoint), device=DEVICE)
        after, _ = loaded.predict(check_obs, deterministic=True)
        np.testing.assert_allclose(before, after, rtol=1e-5, atol=1e-6)
        assert np.isfinite(after).all()
        print("Saved/reloaded actions agree:", after)
        del loaded
    finally:
        check_env.close()


## 12. Start a longer run when the checks pass

First set `RUN_LONG_TRAINING=True` and `TARGET_TOTAL_TIMESTEPS=100_000` for a representative benchmark. Inspect speed, logs and one video. If the pipeline behaves correctly and time allows, increase the target to 250,000, 500,000 or 1,000,000 and rerun this cell. Those are inspection points, not promises of convergence.

`TARGET_TOTAL_TIMESTEPS` is the cumulative target. If the model has already trained for 250,000 transitions and you enter 500,000, this asks for roughly 250,000 more. PPO may round up to complete a rollout. `reset_num_timesteps=False` preserves the counter.

The cell is opt-in so **Run all** does not accidentally launch hours of training. Interrupting a cell normally triggers `finally` and saves the current weights; a hard runtime loss may not, which is why periodic checkpoints matter. A mid-update save supports practical continuation but not bit-identical replay.

`progress_bar=False` intentionally avoids the Rich/IPython recursive-output failure observed at validation in Colab. SB3 text tables remain enabled. Keep evaluation every 100,000 and saving every 25,000. A checkpoint can be saved mid-rollout; its transition counter does not imply that all counted samples already contributed to an optimizer update.


In [ ]:
# Use ordinary SB3 text logs: Rich progress-bar output recursed in Colab at validation.
RUN_LONG_TRAINING = False  # Change to True after inspecting the short check.
TARGET_TOTAL_TIMESTEPS = 100_000  # Initial comparison; then 250_000 / 500_000 if time allows.

if RUN_LONG_TRAINING:
    if DEVICE != "cuda":
        print("Running on CPU; measure throughput before committing to a large budget.")
    remaining = max(0, TARGET_TOTAL_TIMESTEPS - model.num_timesteps)
    if remaining:
        callback = RacingCallback(save_every=25_000, evaluate_every=100_000)
        started = time.perf_counter()
        start_steps = model.num_timesteps
        try:
            model.learn(
                total_timesteps=remaining, callback=callback,
                reset_num_timesteps=False, tb_log_name="ppo", progress_bar=False,
            )
        finally:
            saved = save_checkpoint(model, "latest")
            elapsed = time.perf_counter() - started
            completed = model.num_timesteps - start_steps
            print("Checkpoint:", saved)
            print(f"This segment: {completed:,} transitions, {elapsed / 60:.1f} minutes")
            if completed:
                print(f"Measured segment throughput (including callbacks): {completed / elapsed:.1f}/s")
    else:
        print("Target already reached. Increase the cumulative target to continue.")
else:
    print("Long training is off. Inspect the checks, then enable it explicitly.")


## 13. Read the training logs

`rollout/ep_rew_mean` is recent training reward, not unseen-track skill. `ep_len_mean` counts policy decisions, not raw simulator steps. `time/fps` counts policy transitions and includes overhead; its meaning is not lap time.

`train/approx_kl` measures approximate policy change; persistent large changes warrant inspecting the learning rate and update count. `clip_fraction` measures how often ratios exceeded the clip range. `entropy_loss` is negative entropy under SB3's convention; it is not the same as crash rate. `value_loss` measures critic target error. `explained_variance` near 1 suggests better value-target prediction; negative values suggest a poor predictor, but none of these metrics alone proves good driving. `train/std` summarizes learned noise-weight scales for gSDE; it is not the effective action standard deviation at every observation.

After validation starts, watch `validation/penalized_seconds` and `validation/completion_rate`. Plot training and held-out results together to detect a generalization gap. TensorBoard logs below are local to this runtime; the next cell copies them to Drive after training.

An entropy-loss entry with a zero coefficient is expected. No metric alone proves a completed lap.


In [ ]:
# TensorBoard opens an inline dashboard. Run this after some training logs exist.
%load_ext tensorboard
%tensorboard --logdir $LOG_DIR


In [ ]:
# Back up this session's logs after a training segment.
import shutil
if USE_DRIVE:
    shutil.copytree(LOG_DIR, RUN_DIR / "tensorboard_logs", dirs_exist_ok=True)
    print("Logs copied to Drive")
else:
    print("Local logs:", LOG_DIR)


## 14. Compare training tracks with all 20 validation tracks

This cell is opt-in because driving 25 complete episodes takes time. It evaluates the best checkpoint from the five-track selection subset, if available, otherwise the current model. It does not update weights or automatically switch to another checkpoint based on the final test.

Training-track results are a diagnostic, not the model-selection score. Good training performance with weak validation suggests a generalization problem. Poor results on both groups suggest checking the pipeline and learning before assuming overfitting. A larger training pool is a controlled follow-up experiment, not a guaranteed fix.

The full 20-track validation audit is still validation: using it to change settings is allowed, but repeated selection can overfit this set too. Save the untouched final test for the end.


In [ ]:
best_path = RUN_DIR / "best_validation.zip"
if best_path.exists():
    evaluation_model = PPO.load(str(best_path), device=DEVICE)
    print("Evaluating best_validation.zip at", evaluation_model.num_timesteps, "transitions")
else:
    evaluation_model = model
    print("No selected checkpoint yet; using the current model")

RUN_FULL_VALIDATION = False
if RUN_FULL_VALIDATION:
    train_rows = evaluate_tracks(evaluation_model, TRAINING_SEEDS[:5], split="training")
    validation_rows = evaluate_tracks(evaluation_model, VALIDATION_SEEDS, split="validation")
    report = {
        "timesteps": evaluation_model.num_timesteps,
        "training": summarize(train_rows), "validation": summarize(validation_rows),
        "tracks": train_rows + validation_rows,
    }
    report_path = RUN_DIR / f"validation_audit_{evaluation_model.num_timesteps}.json"
    report_path.write_text(json.dumps(report, indent=2))
    print(json.dumps({k: v for k, v in report.items() if k != "tracks"}, indent=2))
else:
    print("Set RUN_FULL_VALIDATION=True when ready for the 20-track validation audit.")


## 15. Watch the driver inside the notebook

Set `RECORD_VIDEO=True` to record one validation track. Colab displays an MP4 below; it does not open a desktop game window. The video is streamed to a local file to keep memory use modest, then copied to the run folder. Frame timing is for visualization; scoring comes from raw simulator steps.

After long training, rerun the checkpoint-selection cell above before this cell so `evaluation_model` represents the checkpoint you want. An early model may spin, brake or fail to accelerate. Inspect observations/action distributions and allow appropriate training before blaming the network size.


In [ ]:
RECORD_VIDEO = False
if RECORD_VIDEO:
    local_video = Path("/content") / f"{ARCHITECTURE}_preview.mp4" if USE_DRIVE else RUN_DIR / "preview.mp4"
    video_rows = evaluate_tracks(
        evaluation_model, (VALIDATION_SEEDS[0],),
        split="validation", video_path=local_video,
    )
    print(video_rows[0])
    if USE_DRIVE:
        shutil.copy2(local_video, RUN_DIR / "preview.mp4")
    display(Video(str(local_video), embed=True))


## 15a. Inspect controls without changing the trained policy

This optional check uses a separate environment on validation seed 10000. It prints deterministic steering, gas and brake every ten decisions and saves every decision to JSON. It stops after one episode or 200 decisions (at most 400 raw steps), so it is an early-behavior diagnostic, not a full race score. Use section 14 for full races.

Unlike the previous diagonal-Gaussian diagnostic, this check works with gSDE and does not assume that the policy has only three global standard-deviation parameters. It does not train or sample exploration noise. Scored videos and evaluations remain deterministic. If you separately test stochastic gSDE inference, `predict(deterministic=False)` does not automatically resample noise; see the [SB3 PPO instructions](https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html).


In [ ]:
def inspect_controls(policy, track_seed=10000, max_decisions=200):
    """Observe an early rollout without updating weights or reusing train_env."""
    if track_seed not in VALIDATION_SEEDS:
        raise ValueError("Use a validation seed for this diagnostic")
    env = build_vec_env((track_seed,), n_envs=1, random_order=False)
    rows = []
    total_reward = 0.0
    episode_ended = False
    raw_steps = 0
    try:
        observation = env.reset()
        for decision in range(max_decisions):
            action, _ = policy.predict(observation, deterministic=True)
            steering, gas, brake = (float(x) for x in action[0])
            row = {"decision": decision, "steering": steering, "gas": gas, "brake": brake}
            rows.append(row)
            if decision % 10 == 0:
                print(f"Decision {decision}: steering={steering:.3f}, gas={gas:.3f}, brake={brake:.3f}")
            observation, rewards, dones, infos = env.step(action)
            total_reward += float(rewards[0])
            raw_steps = int(infos[0]["raw_steps"])
            if dones[0]:
                episode_ended = True
                break
    finally:
        env.close()
    report = {
        "timesteps": int(policy.num_timesteps), "track_seed": int(track_seed),
        "episode_ended": episode_ended, "raw_steps": raw_steps,
        "observed_reward": total_reward, "actions": rows,
    }
    path = RUN_DIR / f"controls_{policy.num_timesteps}_{track_seed}.json"
    path.write_text(json.dumps(report, indent=2))
    print("Episode ended:", episode_ended, "Raw steps:", raw_steps)
    print("Control report:", path)
    return report


In [ ]:
RUN_CONTROL_DIAGNOSTICS = False
if RUN_CONTROL_DIAGNOSTICS:
    control_report = inspect_controls(model)


## 16. Final test and public tracks: run after choosing the model

Leave `RUN_FINAL_TEST=False` until the architecture/settings/checkpoint have been selected using validation. This uses all 20 untouched test tracks. If you change the model based on these results, the set is no longer an untouched test.

Public tracks are reported separately, with three deterministic runs each to match the challenge's repetition count. Fixed seeds plus a deterministic policy can produce identical repetitions, so this is not nine independent generalization examples. Compare the organizer's exact timing/loader rules before submission.


In [ ]:
RUN_FINAL_TEST = False
RUN_PUBLIC_TRACKS = False
if RUN_FINAL_TEST:
    rows = evaluate_tracks(evaluation_model, TEST_SEEDS, split="test")
    report = {"timesteps": evaluation_model.num_timesteps, "summary": summarize(rows), "tracks": rows}
    (RUN_DIR / "final_test.json").write_text(json.dumps(report, indent=2))
    print("Final test:", report["summary"])
if RUN_PUBLIC_TRACKS:
    repeated_seeds = tuple(seed for seed in PUBLIC_SEEDS for _ in range(3))
    rows = evaluate_tracks(evaluation_model, repeated_seeds, split="public")
    (RUN_DIR / "public_tracks.json").write_text(json.dumps(rows, indent=2))
    for seed in PUBLIC_SEEDS:
        print(seed, summarize([row for row in rows if row["track_seed"] == seed]))


## 17. Export weights and understand what has been saved

`latest.zip` contains the resumable SB3 training checkpoint. `best_validation.zip` is the selected checkpoint from periodic validation. Keep both. The cell below also exports a `.pth` bundle containing the selected policy's CPU tensors, its settings and its timestep count.

**The `.pth` bundle is not a complete official submission adapter.** Reconstructing this SB3 policy still requires the encoder class and the exact preprocessing, frame history reset, Gaussian-mean action handling and repeat timing. The organizer's loader interface is not supplied here. Do not submit an unexplained weights file and assume it will run. Saving a TorchScript model or a standalone actor later must reproduce this policy's deterministic actions across an entire rollout.

No pretrained CarRacing weights are downloaded. These files contain only the policy you train in this notebook.


In [ ]:
EXPORT_WEIGHTS = False
if EXPORT_WEIGHTS:
    weights_path = RUN_DIR / "selected_policy_weights.pth"
    torch.save({
        "policy_state_dict": {k: v.detach().cpu() for k, v in evaluation_model.policy.state_dict().items()},
        "config": CONFIG,
        "num_timesteps": evaluation_model.num_timesteps,
        "sb3_version": str(sb3.__version__),
        "gymnasium_version": str(gym.__version__),
    }, weights_path)
    print("Weights bundle:", weights_path)
print("Run folder:", RUN_DIR)
print("Checkpoints:", [path.name for path in RUN_DIR.glob("*.zip")])


## 18. Resume, compare and troubleshoot

**Resume:** reconnect to a GPU runtime; run installation/imports; mount Drive; use the same experiment name and settings; run wrapper/encoder/callback definitions; set `RESUME_FROM` to `str(RUN_DIR / "latest.zip")` in section 9; skip the smoke update if unnecessary; increase the cumulative target in section 12. No need to download weights from Drive to your laptop first.

**Compare encoders:** run the other notebook separately with the same training pool, run seed, PPO settings and validation tracks. Compare both wall-clock compute and transition counts; a deeper network may consume more time per transition. Start each architecture from scratch. Select by validation penalized time and completion rate, not by parameter count or a single fast lap. If compute allows, repeat a promising setup with another run seed in a new folder.

**GPU unavailable:** choose an available NVIDIA GPU runtime, then rerun imports to confirm `Device: cuda`. CPU can run checks, but benchmark before long training. Box2D physics remains CPU-bound.

**Box2D/SWIG install error:** rerun the SWIG installation cell first. If the error specifically says a C++ compiler is missing, use a separate Colab cell `!apt-get update -qq` followed by `!apt-get install -y -qq build-essential`, then retry the dependencies. Restart only if requested and rerun definitions.

**Drive mount failure:** retry your Drive authentication; do not silently train for hours into temporary storage. For a temporary setup check only, `USE_DRIVE=False` is available.

**NaN, shape or loading error:** stop and read the first failing assertion. Confirm `(n_envs,4,84,84)`, uint8 before SB3, no double division by 255, the matching encoder and settings, and the pinned package versions. Restarting a cell does not automatically undo previous training.

**All tracks DNF:** inspect video, action means/spread and training logs. A 1024-step check cannot establish performance. If training tracks improve and validation does not, test more training diversity. If neither improves, inspect learning and pipeline correctness before increasing network depth.

**Clean up:** run `train_env.close()` when finished. Save/check Drive checkpoints before disconnecting the runtime. Saving this notebook alone does not save model state.

**Matched comparison:** compare against `ppo_cnn_gsde_recovery.ipynb`, not the original ordinary-Gaussian basic-CNN notebook. Keep gSDE, PPO, preprocessing, rewards and track pools fixed. The same run seed is not a claim of identical sampled experience across different architectures.


## References and limits of the architecture choice

- [PPO paper](https://arxiv.org/abs/1707.06347): clipped on-policy actor-critic learning; not a CarRacing winning guarantee.
- [SB3 PPO documentation](https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html): the actual PPO implementation used here.
- [SB3 custom feature extractors](https://stable-baselines3.readthedocs.io/en/master/guide/custom_policy.html): how our CNN is connected to the policy.
- [Gymnasium CarRacing](https://gymnasium.farama.org/environments/box2d/car_racing/): observations, actions and reward.
- [IMPALA paper](https://arxiv.org/abs/1802.01561): source of the residual encoder family; this notebook does not use the IMPALA learning algorithm.
- [Procgen paper](https://cdn.openai.com/procgen.pdf): motivation for varied levels and separate unseen evaluation. Its results do not prove that 100 CarRacing tracks are sufficient or that this encoder wins this race.
- [Colab FAQ](https://research.google.com/colaboratory/faq.html): hardware and session limits vary.

The notebook is an explainable baseline experiment. There is no measured claim here that either architecture is the fastest. Run the same protocol and let held-out results decide.
